In [ ]:
import pandas as pd, numpy as np, glob
from collections import Counter

files = sorted(glob.glob("data_preprocessed/*_combined.npz"))
for f in files:
    arr = np.load(f)
    y = arr["y"]
    print(f"{f.split('/')[-1]:<20} → {Counter(y)}")

In [ ]:
for f in sorted(glob.glob("data_preprocessed/*_combined.npz")):
    arr = np.load(f)
    print(f"{f.split('/')[-1]} → X: {arr['X'].shape}, y: {arr['y'].shape}, labels: {np.unique(arr['y'])}")


In [ ]:
import os

def check_folder_exists(folder_path):
    if os.path.exists(folder_path) and os.path.isdir(folder_path):
        print(f"The folder '{folder_path}' exists.")
    else:
        print(f"The folder '{folder_path}' does not exist.")

# Example usage
folder_path = "WESAD"
check_folder_exists(folder_path)


In [ ]:
import pickle
import numpy as np
from pathlib import Path
def debug_subject(pkl_path):
    """Debug: print actual shapes in pickle file."""
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    
    signals = data["signal"]
    labels = np.array(data["label"], dtype=np.int32)
    chest = signals["chest"]
    wrist = signals.get("wrist", {})
    
    print(f"\nDEBUG {pkl_path.stem}:")
    print(f"  labels length: {len(labels)}")
    print(f"  chest.ACC shape: {np.array(chest.get('ACC', [])).shape}")
    print(f"  chest.ECG shape: {np.array(chest.get('ECG', [])).shape}")
    print(f"  chest.EDA shape: {np.array(chest.get('EDA', [])).shape}")
    print(f"  wrist.ACC shape: {np.array(wrist.get('ACC', [])).shape}")
    print(f"  wrist.EDA shape: {np.array(wrist.get('EDA', [])).shape}")

# Test on one file
debug_subject(Path("WESAD/S2.pkl"))

In [1]:
import numpy as np
from pathlib import Path

# Check one test subject
test_file = Path("data_preprocessed/S2_combined.npz")
with np.load(test_file) as data:
    X, y = data['X'], data['y']
    print(f"S2 shape: {X.shape}")
    print(f"S2 labels: {np.unique(y, return_counts=True)}")
    print(f"S2 label distribution: {[(i, (y==i).sum()) for i in range(4)]}")

# Check another
test_file2 = Path("data_preprocessed/S3_combined.npz")
with np.load(test_file2) as data:
    X2, y2 = data['X'], data['y']
    print(f"\nS3 shape: {X2.shape}")
    print(f"S3 labels: {np.unique(y2, return_counts=True)}")
    
# Check if any data is identical (corruption indicator)
print(f"\nFirst 5 samples S2 == First 5 samples S3: {np.allclose(X[:5], X2[:5])}")

S2 shape: (1265, 320, 8)
S2 labels: (array([0, 1, 2, 3]), array([903, 128,  75, 159]))
S2 label distribution: [(0, np.int64(903)), (1, np.int64(128)), (2, np.int64(75)), (3, np.int64(159))]

S3 shape: (1351, 320, 8)
S3 labels: (array([0, 1, 2, 3]), array([979, 133,  78, 161]))

First 5 samples S2 == First 5 samples S3: False


In [2]:
from pathlib import Path
import numpy as np

DATA_DIR = Path("data_preprocessed")
files = sorted(DATA_DIR.glob("*_combined.npz"))

print(f"Found {len(files)} files")

# Try loading just one file to see if disk is accessible
test_file = files[0]
print(f"Attempting to load {test_file.name}...")

try:
    with np.load(test_file) as data:
        X = data['X']
        y = data['y']
        print(f"Successfully loaded: {X.shape}, {y.shape}")
        print(f"File size on disk: {test_file.stat().st_size / (1024**2):.1f} MB")
except Exception as e:
    print(f"Failed to load: {e}")

# Check if files are on OneDrive (which can cause hangs)
print(f"\nFile path: {test_file}")
if "OneDrive" in str(test_file):
    print("⚠️ WARNING: Files are on OneDrive!")
    print("OneDrive can cause hangs during file operations.")
    print("Solution: Copy WESAD and data_preprocessed to local drive (C:\\Users\\indra\\...)")

Found 15 files
Attempting to load S10_combined.npz...
Successfully loaded: (1143, 320, 8), (1143,)
File size on disk: 4.0 MB

File path: data_preprocessed\S10_combined.npz


In [4]:
# Test: Just one forward pass on GPU
import torch
import torch.nn as nn

# Create dummy batch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(32, 320, 8).to(DEVICE)  # (batch, time, channels)

# Build model
model = build_model(
    input_channels=8,
    cnn_ch=64,
    gru_hidden=128,
    gru_layers=2,
    attn_heads=2,
    dropout=0.3,
    num_classes=NUM_CLASSES,
).to(DEVICE)

print("Model built and on GPU")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

# Try forward pass
print("\nAttempting forward pass...")
import time

t0 = time.time()
with torch.no_grad():
    output = model(x)
t1 = time.time()

print(f"✓ Forward pass completed in {t1-t0:.2f} seconds")
print(f"Output shape: {output.shape}")

NameError: name 'build_model' is not defined